# Product Pairs ETL

## Purpose
Identify frequently co-purchased product pairs for recommendation systems, cross-selling strategies, and inventory optimization.

## Input → Output
* **Source:** `big_data.silver.order_products` 
* **Target:** `big_data.gold.ft_product_pairs`
* **Primary Key:** (product_id_1, product_id_2)

## Transformations
1. Self-Join to Build Product Pairs - Self-join order_products on order_id, filter product_id_1 < product_id_2 (avoid duplicates)
2. Aggregate and Filter - GROUP BY (product_id_1, product_id_2), COUNT occurrences as times_bought_together, filter >= 5 co-purchases, add timestamp

## Data Quality
* **Technical:** Pair count > 1M, NOT NULL (product_id_1, product_id_2, times_bought_together), All times_bought_together >= 5
* **Business:** product_id_1 < product_id_2 (no reversed duplicates), Distribution analysis

## Persistence
Writes to Delta table **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
# PySpark imports
from pyspark.sql import functions as F

In [0]:
# Schema configuration
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Source table
source_table = "order_products"

# Target table (fact table with ft_ prefix)
target_table = "ft_product_pairs"

# Primary Key columns (composite key)
primary_key_columns = ["product_id_1", "product_id_2"]

# Critical columns (NOT NULL required)
critical_columns = ["product_id_1", "product_id_2", "times_bought_together"]

# Validation thresholds
expected_metrics = {
    "min_pairs": 1_000_000,  # Expect at least 1M pairs
    "min_times_together": 5   # Filter threshold
}

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Configuration:")
print(f"  Source schema: {silver_schema}")
print(f"  Source table: {source_table}")
print(f"  Target: {gold_schema}.{target_table}")
print(f"  Primary Key: {', '.join(primary_key_columns)}")

### TRANSFORMATION

In [0]:
print("Step 1: Building product pairs via self-join...")

# Load order_products
order_products = spark.table(f"{silver_schema}.{source_table}")

print(f"  Order-Products loaded: {order_products.count():,} rows")

# Self-join to find product pairs in the same order
# Filter: product_id_1 < product_id_2 to avoid duplicate pairs (A,B) and (B,A)
pairs = order_products.alias("a").join(
    order_products.alias("b"),
    (F.col("a.order_id") == F.col("b.order_id")) & 
    (F.col("a.product_id") < F.col("b.product_id"))
).select(
    F.col("a.product_id").alias("product_id_1"),
    F.col("b.product_id").alias("product_id_2"),
    F.col("a.order_id")
)

print(f"  Product pairs identified: {pairs.count():,} raw pairs")

In [0]:
print("Step 2: Aggregating and filtering product pairs...")

# Aggregate by product pair and count occurrences
product_pairs_gold = pairs.groupBy("product_id_1", "product_id_2").agg(
    F.count("order_id").alias("times_bought_together")
).filter(
    F.col("times_bought_together") >= expected_metrics["min_times_together"]
).withColumn(
    "_gold_timestamp", F.current_timestamp()
).orderBy(
    F.desc("times_bought_together")
)

print(f"  Filtered pairs (>= {expected_metrics['min_times_together']} occurrences): {product_pairs_gold.count():,}")
print("\nPreview - Top 10 Product Pairs:")
product_pairs_gold.show(10, truncate=False)

In [0]:
# Create final DataFrame for validation and persistence
df_result = product_pairs_gold

print(f"\nFinal DataFrame 'df_result' created: {df_result.count():,} rows")
print("\nReady for validation and persistence")

### DATA QUALITY

In [0]:
print_validation_header("Product Pairs - Technical Validations")

# Initialize validation flag
validation_technical = True

# 1. Row count check
pair_count = df_result.count()
print(f"\nProduct pair count: {pair_count:,}")
print(f"Expected: >= {expected_metrics['min_pairs']:,}\n")

if pair_count >= expected_metrics["min_pairs"]:
    status = "PASS"
    msg = f"Pair count ({pair_count:,}) >= {expected_metrics['min_pairs']:,}"
else:
    status = "FAIL"
    msg = f"Pair count ({pair_count:,}) < {expected_metrics['min_pairs']:,}"
    validation_technical = False
print_check_result("PAIR COUNT (>= 1M)", status, msg)

# 2. NOT NULL checks
print("\n2. NOT NULL Validations:")
status, failed, msg = check_not_null(df_result, critical_columns)
print_check_result(f"NOT NULL ({len(critical_columns)} columns)", status, msg, failed)
if status == "FAIL":
    validation_technical = False

# 3. Filter threshold check
print("\n3. Filter Threshold Validation:")
below_threshold = df_result.filter(
    F.col("times_bought_together") < expected_metrics["min_times_together"]
).count()

if below_threshold == 0:
    status = "PASS"
    msg = f"All pairs have times_bought_together >= {expected_metrics['min_times_together']}"
else:
    status = "FAIL"
    msg = f"{below_threshold} pairs below threshold ({expected_metrics['min_times_together']})"
    validation_technical = False
print_check_result(f"FILTER THRESHOLD (>= {expected_metrics['min_times_together']})", status, msg, below_threshold)

total_rows = pair_count

print("\n" + "="*60)
if validation_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("Product Pairs - Business Validations")

# Initialize business validation flag
validation_business = True

# 1. No reversed duplicates (product_id_1 < product_id_2)
print("\n1. Business Rule - No Reversed Duplicates:")
reversed_pairs = df_result.filter(
    F.col("product_id_1") >= F.col("product_id_2")
).count()

if reversed_pairs == 0:
    status = "PASS"
    msg = "All pairs satisfy product_id_1 < product_id_2"
else:
    status = "FAIL"
    msg = f"{reversed_pairs} pairs have product_id_1 >= product_id_2"
    validation_business = False
print_check_result("NO REVERSED DUPLICATES", status, msg, reversed_pairs)

# 2. Distribution analysis
print("\n2. Business Rule - Distribution Analysis:")
top_pairs = df_result.limit(5).collect()
print("\n  Top 5 Product Pairs by Co-Purchase Frequency:")
for pair in top_pairs:
    print(f"    - Product {pair['product_id_1']} + Product {pair['product_id_2']}: {pair['times_bought_together']:,} times")

status = "PASS"
msg = f"Distribution looks reasonable ({total_rows:,} unique pairs)"
print_check_result("DISTRIBUTION ANALYSIS", status, msg)

print("\n" + "="*60)
if validation_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta table using UTILS function
if validation_passed:
    persist_to_delta(df_result, f"{gold_schema}.{target_table}")
    print("\nNext Step: Use for recommendation systems and cross-selling")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")